# P_beta across a mid-simulation pd switch

Run top to bottom. Seeds 1-10 are the tuning set, seeds 11-30 are held out and
scored exactly once.

**Before running:** `set_pd.m` must be in `matlab/`, and this notebook must be
launched from the workspace root (the folder containing `helpers.py`).

## 1. Environment check

In [ ]:
%matplotlib inline

import sys, os
print(sys.executable)
print(os.getcwd())

import matlab.engine
print("matlab.engine ok")

`sys.executable` must be `/opt/anaconda3/bin/python`.
`os.getcwd()` must contain `helpers.py` and `bgn_vars.mat`.

## 2. Imports and MATLAB engine

Takes about 15 s. Leave the engine alive for the whole session.

In [ ]:
import importlib
import numpy as np
import matplotlib.pyplot as plt

import helpers
import pd_switch as ps
importlib.reload(ps)

eng = ps.start_engine()
print("engine up")

## 3. Simulate

30 brain realizations: 2 s warm-up (discarded) + 5 s at pd = 0.3 + 5 s at pd = 0.7.

About 55 s per seed, so ~28 minutes on a cold cache. Results are cached to
`runs_switch/` as they complete, so an interruption costs only the run in flight,
and re-running this cell afterwards is instant.

`run_seeds` caches on seed number alone. If you change `warmup_ms`, `hold_ms`,
`freq` or `amp`, pass a different `out_dir` rather than reusing this one.

In [ ]:
res_all, zooms = ps.run_seeds(eng, range(1, 31), helpers.calculate_pb)

res_tune = {k: v for k, v in res_all.items() if k <= 10}
res_held = {k: v for k, v in res_all.items() if k > 10}
print(f"{len(res_tune)} tuning seeds, {len(res_held)} held out")

## 4. How big is the change?

`gap` is the percentage rise in P_beta from the pd = 0.3 segment to the pd = 0.7
segment. `separation` is that rise measured in units of within-segment noise
(Cohen's d) -- it is what decides whether the two states can be told apart.

Windows that straddle the switch are excluded from both groups.

In [ ]:
gaps, seps = ps.summarize(res_all)

print(f"gap         {gaps.mean():5.1f}% +/- {gaps.std(ddof=1)/np.sqrt(len(gaps)):.1f}")
print(f"separation  {seps.mean():5.2f} sd +/- {seps.std(ddof=1)/np.sqrt(len(seps)):.2f}")
print(f"worst seed  {seps.min():5.2f} sd  (seed {list(res_all)[int(np.argmin(seps))]})")

## 5. The figure

Thin lines are individual brains, the heavy line is the mean, the band is +/- 1 sd
across seeds. The grey span marks windows containing data from both states.

In [ ]:
fig = ps.plot_seeds(res_all)
fig

One seed on its own, with the before/after means drawn in.

In [ ]:
fig = ps.plot_pb(*res_all[1])
fig

## 6. The switch is continuous

`set_pd` rewrites only `pd` inside `bgn_vars.mat`, leaving every membrane voltage
and gating variable untouched, so the simulation carries straight through. This is
the direct check: +/- 5 ms of raw GPi voltage around the switch instant. There
should be no step, no spike, and no change in the character of the spiking.

In [ ]:
fig = ps.plot_zoom(zooms[1])

## 7. Detecting the switch

Two online detectors. Both see only the windows available so far, and both report
the time at which the decision could actually be made -- the end of the window
that triggered it, not its midpoint.

- `detect_cp`: for every possible split of the history, how many standard errors
  apart are the two halves? Fire when the best split exceeds `thresh`.
- `detect_kmeans`: k-means with k=2 on the history, gated on centroid separation
  and on the high cluster forming a near-contiguous run at the end.

Tune on seeds 1-10 only.

In [ ]:
for thresh in (5, 5.5, 6, 6.5, 7, 7.5, 8, 9, 10):
    r = ps.evaluate(res_tune, ps.detect_cp, thresh=thresh)
    ps.report(f"changepoint thresh={thresh}", r)

Pick a threshold in the middle of the clean run, not at its edge. False alarms
appear below it and misses above it; the midpoint is the furthest point from both.

In [ ]:
for gap in (0.10, 0.13, 0.16):
    for breaks in (4, 6, 8):
        r = ps.evaluate(res_tune, ps.detect_kmeans, min_gap=gap, max_breaks=breaks)
        ps.report(f"kmeans gap={gap} breaks={breaks}", r)

## 8. Held-out result

The configurations below were chosen on seeds 1-10. Scoring them on seeds 11-30
gives an honest performance estimate.

**Run this once.** If you re-tune after seeing it, these seeds stop being held out
and the number stops meaning anything -- generate seeds 31-50 for the next round.

In [ ]:
CP_THRESH = 7.0
KM_GAP, KM_BREAKS = 0.13, 6

ps.report("changepoint", ps.evaluate(res_held, ps.detect_cp, thresh=CP_THRESH))
ps.report("kmeans", ps.evaluate(res_held, ps.detect_kmeans,
                                min_gap=KM_GAP, max_breaks=KM_BREAKS))

## 9. Known limits

- `calculate_pb` sums FFT **bin indices** 13:35, which equal frequencies in Hz only
  when the window is 1000 ms. That sets a hard 1 s floor on detection latency.
- The first ~1.5 s of any simulation is the model settling from its initial
  conditions, not the patient. A 2 s warm-up is simulated and discarded. Some slow
  drift remains; `warmup_ms=4000` is untested and may reduce false alarms.
- `detect_cp` tests whether the later half is **higher**, so it fires on rising
  intensity only. A 0.7 -> 0.3 transition would never trigger it.
- Only the 0.3 -> 0.7 pair has been tested, in one direction.
- 20 held-out seeds is not enough to call one detector significantly better than
  the other.

In [ ]:
eng.quit()
print("engine closed")